# Behavioral Bot Detection — Exploration

Second phase of the project: instead of reading the user-agent, look at **how a session behaves**.
This targets bots that disguise themselves with a normal browser UA (same string as Chrome/Safari),
which slip past the User-Agent-based approach entirely.

**This notebook is exploratory**, it does not produce an automatic upload like `bot_detection.py`:
the goal here is to validate features and models on real data, look at the results, and only later
think about integrating this with the existing pipeline.

**Before running this against your own AEP instance**, edit `XDM_NAMESPACE` and `DATASETS` in
`src/bot_detection.py` — this notebook imports them from there. You'll also need to adjust the field
paths inside the session-reconstruction query below (`eventtype`, `pagedetails.full_url`, ...) to
match your own XDM schema — see the "Field paths" note before the query.

Dependencies:
```
pip install -r ../requirements.txt
```

## 1. Imports and configuration

In [ ]:
import os
import sys
import re
import getpass
import math

import numpy as np
import pandas as pd
import psycopg2
from psycopg2 import sql

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from bot_detection.aep_pipeline import XDM_NAMESPACE, DATASETS, DEFAULT_BOT_TABLE, resolve_bot_table

try:
    from dotenv import find_dotenv, load_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f"Loaded variables from {env_path}")
    else:
        print("No .env file found: falling back to getpass prompts below.")
except ImportError:
    print("python-dotenv not installed: falling back to getpass prompts below.")

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

In [ ]:
AEP = {
    "host": os.getenv("AEP_HOST") or input("AEP host: "),
    "port": int(os.getenv("AEP_PORT", "80")),
    "dbname": os.getenv("AEP_DBNAME", "prod:all"),
    "user": os.getenv("AEP_USER") or input("AEP user: "),
    "password": os.getenv("AEP_PASSWORD") or getpass.getpass("AEP password: "),
    "sslmode": "require",
}
print("AEP config ready:", AEP["host"])

In [ ]:
# Opened once, reused by every cell below (type check, session query, known-bots lookup).
conn = psycopg2.connect(**AEP)
print("Connected.")

## 2. Parameters

Start with **one dataset and a few days**, on purpose: the session-reconstruction query uses window
functions over raw hits (not just distinct user-agents), so it's heavier than the queries in
`bot_detection_exploration.ipynb`. Validate the logic before scaling up.

In [ ]:
DATASET = DATASETS[0]     # a single dataset for this exploration
DAYS = 3                  # short window: window functions over every raw hit, not just distinct UAs
SESSION_GAP_SECONDS = 1800  # 30 minutes — check your actual session-timeout setting in CJA

# AEP Query Service caps results at 50,000 rows if you don't specify an explicit LIMIT.
EXTRACTION_LIMIT = 10_000_000

print(f"Dataset: {DATASET} | window: {DAYS} days | session gap: {SESSION_GAP_SECONDS}s")

BOT_TABLE = resolve_bot_table(conn, expected=DEFAULT_BOT_TABLE)
print("Using bot table:", BOT_TABLE)

## 3. Session reconstruction

AEP doesn't have a native `session_id` — CJA computes it downstream at query time, it's not part of
the raw data. We rebuild session boundaries ourselves with a standard "sessionization" technique:
order hits by visitor and timestamp, flag a new session whenever the gap since the previous hit
exceeds the threshold, then a running sum of those flags gives a session counter.

**Note on Spark SQL**: `UNIX_TIMESTAMP()` instead of `EXTRACT(EPOCH FROM ...)` for the gap
calculation — the standard Postgres syntax raises a type error on this engine.

**Field paths — adapt these to your own schema.** Every `{ns}` below is `XDM_NAMESPACE` from
`bot_detection.py`. The paths after it (`.id`, `.pagedetails.full_url`, `.device.useragent`,
`.pagedetails.dom_referrer`, `.events.events_cartadd`) are specific to how this project's original
schema happened to be laid out — yours will very likely use different field names. Use
`SHOW COLUMNS IN <dataset>` in the AEP Query Service UI to find your real field paths, then edit the
query below accordingly, along with `eventtype` (used as-is, not namespaced, since it's typically a
standard XDM field) and the `cart_add_flag` value comparison (`= true`) if your field is an integer
or string instead of a boolean — check with the type-verification cell right below.

### Preliminary check: what type is your "cart add" field?

Before using `= true` in the big query's `CASE WHEN`, check whether your equivalent field is really
boolean, or an integer (`0`/`1`) or a string — otherwise you'll hit the same kind of type error
already seen with `EXTRACT(EPOCH FROM ...)`.

In [ ]:
with conn.cursor() as cur:
    cur.execute(sql.SQL(
        "SELECT DISTINCT {ns}.events.events_cartadd, "
        "typeof({ns}.events.events_cartadd) "
        "FROM {dataset} LIMIT 10"
    ).format(ns=sql.SQL(XDM_NAMESPACE), dataset=sql.Identifier(DATASET)))
    for value, dtype in cur:
        print(f"value: {value!r}  |  type: {dtype}")

If the field comes back **boolean**, the query below works as-is (`cart_add_flag = true`). If it's an
integer or string, change `sum(CASE WHEN cart_add_flag = true THEN 1 ELSE 0 END)` in the next cell to
`sum(CASE WHEN cart_add_flag = 1 THEN 1 ELSE 0 END)` (integer) or
`sum(CASE WHEN cart_add_flag = 'true' THEN 1 ELSE 0 END)` (string).

In [ ]:
def build_sessionization_query(dataset, days, gap_seconds, limit):
    ns = sql.SQL(XDM_NAMESPACE)
    return sql.SQL("""
        WITH events AS (
            SELECT
                {ns}.id AS visitor_id,
                timestamp,
                eventtype AS event_type,
                {ns}.pagedetails.full_url AS page_url,
                {ns}.device.useragent AS useragent,
                {ns}.pagedetails.dom_referrer AS referrer,
                {ns}.events.events_cartadd AS cart_add_flag
            FROM {dataset}
            WHERE timestamp >= current_date - interval {days}
              AND {ns}.id IS NOT NULL
            LIMIT {limit}
        ),
        gapped AS (
            SELECT
                *,
                UNIX_TIMESTAMP(timestamp) - UNIX_TIMESTAMP(
                    LAG(timestamp) OVER (PARTITION BY visitor_id ORDER BY timestamp)
                ) AS gap_seconds
            FROM events
        ),
        flagged AS (
            SELECT
                *,
                CASE WHEN gap_seconds IS NULL OR gap_seconds > {gap} THEN 1 ELSE 0 END AS new_session_flag
            FROM gapped
        ),
        sessioned AS (
            SELECT
                *,
                visitor_id || '_' || SUM(new_session_flag) OVER (
                    PARTITION BY visitor_id ORDER BY timestamp
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS session_id
            FROM flagged
        )
        SELECT
            session_id,
            visitor_id,
            count(*) AS n_hits,
            min(timestamp) AS session_start,
            max(timestamp) AS session_end,
            count(DISTINCT page_url) AS unique_pages,
            sum(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) AS n_clicks,
            sum(CASE WHEN cart_add_flag = true THEN 1 ELSE 0 END) AS n_cart_adds,
            max(useragent) AS useragent,
            max(referrer) AS referrer
        FROM sessioned
        GROUP BY session_id, visitor_id
        LIMIT {limit}
    """).format(
        ns=ns,
        dataset=sql.Identifier(dataset),
        days=sql.Literal(f"{days} day"),
        gap=sql.Literal(gap_seconds),
        limit=sql.Literal(limit),
    )


print(build_sessionization_query(DATASET, DAYS, SESSION_GAP_SECONDS, EXTRACTION_LIMIT).as_string(conn))

### Run it

In [ ]:
with conn.cursor() as cur:
    print(f"Sessionizing {DATASET} over {DAYS} days...")
    cur.execute(build_sessionization_query(DATASET, DAYS, SESSION_GAP_SECONDS, EXTRACTION_LIMIT))
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()

sessions = pd.DataFrame(rows, columns=cols)
print(f"Sessions reconstructed: {len(sessions):,}")
sessions.head()

### Sanity check

Before building features on top: do the durations and counts make sense? Negative or absurdly long
sessions point to a problem in the sessionization (e.g. timezone, or a miscalculated gap) worth
fixing before anything downstream.

In [ ]:
sessions["duration_seconds"] = (
    pd.to_datetime(sessions["session_end"]) - pd.to_datetime(sessions["session_start"])
).dt.total_seconds()

print(sessions[["n_hits", "duration_seconds", "unique_pages", "n_clicks", "n_cart_adds"]].describe())

suspicious = sessions[(sessions["duration_seconds"] < 0) | (sessions["duration_seconds"] > SESSION_GAP_SECONDS * 5)]
print(f"\nSessions with suspicious duration: {len(suspicious)} out of {len(sessions)}")

## 4. Feature engineering

The features with the strongest expected discriminative power between bots and humans, roughly in
order:

1. **Peak hits per second** — requests too close together are impossible for a human finger
2. **Path entropy** — a bot iterating systematic paths (`/product/1`, `/product/2`...) has lower
   entropy than human browsing
3. **Interaction ratio** — clicks and cart-adds nearly absent despite many hits is a strong signal.
   We don't have scroll events in this schema, so the signal here is weaker than it would be with
   that too — a "quiet" bot that fires the occasional click to look human can slip past this feature
   alone (one more reason not to rely on any single feature — see ECOD/XGBoost below).
4. **Missing referrer** — a human almost always arrives from somewhere
5. **Pages per session out of range** — either too few (1 hit, typical of a scanner) or too many

Note: the variance of inter-hit time deltas, computed at the individual-hit level, is missing here —
it requires per-row timestamps within a session, not just the aggregate used here. Likely the single
strongest signal, left for a second pass to keep this first exploration simpler.

In [ ]:
def path_entropy(paths: list[str]) -> float:
    """Shannon entropy over the distribution of paths visited in a session.
    Low entropy = few paths repeated systematically (typical of scraping)."""
    if not paths:
        return 0.0
    counts = pd.Series(paths).value_counts(normalize=True)
    return float(-(counts * np.log2(counts)).sum())


sessions["hits_per_second_peak"] = sessions["n_hits"] / sessions["duration_seconds"].clip(lower=1)
sessions["interaction_ratio"] = (
    (sessions["n_clicks"] + sessions["n_cart_adds"])
    / sessions["n_hits"].clip(lower=1)
)
sessions["pages_per_hit"] = sessions["unique_pages"] / sessions["n_hits"].clip(lower=1)
sessions["referrer_missing"] = sessions["referrer"].isna() | (sessions["referrer"] == "")
sessions["single_hit_session"] = sessions["n_hits"] == 1

FEATURES = [
    "n_hits", "duration_seconds", "unique_pages",
    "hits_per_second_peak", "interaction_ratio", "pages_per_hit",
    "referrer_missing", "single_hit_session",
]

sessions[FEATURES].describe()

## 5. Known labels: bots already identified by User-Agent

Reusing the work from the first notebook: sessions whose UA already appears in your bot table become
the positive class for the PU-learning step below. Everything else is "unlabeled", not "confirmed
human" — that's exactly where we want the model to find the disguised bots.

In [ ]:
with conn.cursor() as cur:
    cur.execute(sql.SQL(
        "SELECT DISTINCT lower(trim({ns}.BOT.user_agent)) FROM {t}"
    ).format(ns=sql.SQL(XDM_NAMESPACE), t=sql.Identifier(BOT_TABLE)))
    known_bot_uas = {r[0] for r in cur if r[0]}

conn.close()

sessions["ua_key"] = sessions["useragent"].fillna("").str.strip().str.lower()
sessions["known_bot_by_ua"] = sessions["ua_key"].isin(known_bot_uas).astype(int)

print(f"Sessions with a UA already known as a bot: {sessions['known_bot_by_ua'].sum():,} out of {len(sessions):,} "
      f"({100 * sessions['known_bot_by_ua'].mean():.2f}%)")

## 6. Unsupervised anomaly detection (ECOD)

Answers: "is this session statistically unusual compared to normal traffic?", with no need to know
in advance who's a bot. Useful for catching patterns never seen before. `ECOD` needs no hyperparameter
tuning (unlike IsolationForest) and is interpretable per dimension.

In [ ]:
from pyod.models.ecod import ECOD
from sklearn.preprocessing import RobustScaler

X = sessions[FEATURES].astype(float).fillna(0).values
X_scaled = RobustScaler().fit_transform(X)  # less sensitive to extreme outliers than StandardScaler

ecod = ECOD(contamination=0.02)  # starting estimate: 2% of traffic anomalous, tune on your results
ecod.fit(X_scaled)

sessions["ecod_score"] = ecod.decision_scores_
sessions["ecod_is_anomaly"] = ecod.labels_  # 1 = anomalous per the contamination threshold

print(f"Sessions flagged anomalous by ECOD: {sessions['ecod_is_anomaly'].sum():,} "
      f"({100 * sessions['ecod_is_anomaly'].mean():.2f}%)")
sessions.nlargest(15, "ecod_score")[["session_id", "n_hits", "duration_seconds", "unique_pages",
                                       "interaction_ratio", "known_bot_by_ua", "ecod_score", "useragent"]]

### Do the top anomalies already have a known bot UA?

If the most anomalous sessions per ECOD are mostly already labeled, that's a good sign the features
genuinely capture bot behavior. If many have a "human" UA, those are the most interesting candidates
to check by hand.

In [ ]:
top_anomalies = sessions.nlargest(100, "ecod_score")
print(f"Of the top 100 anomalous sessions: {top_anomalies['known_bot_by_ua'].sum()} already had a known UA, "
      f"{100 - top_anomalies['known_bot_by_ua'].sum()} had a \"human\" UA — these last ones are the most "
      f"interesting candidates to inspect.")

top_anomalies[top_anomalies["known_bot_by_ua"] == 0].head(15)[
    ["session_id", "n_hits", "duration_seconds", "interaction_ratio", "pages_per_hit", "ecod_score", "useragent"]
]

## 7. Supervised classifier with PU learning (XGBoost)

Bots already labeled by UA are the **positive** class; everything else is "unlabeled"
(`scale_pos_weight` compensates for the imbalance). The model learns the behavioral patterns from
already-known bots, then applies them to ALL sessions — including the ones with a clean UA: if the
model assigns a high probability to a session that was never labeled as a bot, it's a disguised-bot
candidate.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

y = sessions["known_bot_by_ua"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

clf = xgb.XGBClassifier(
    scale_pos_weight=pos_weight,
    max_depth=4,
    n_estimators=200,
    learning_rate=0.1,
    eval_metric="aucpr",  # area under precision-recall: more informative than AUC-ROC on imbalanced classes
    random_state=42,
)
clf.fit(X_train, y_train)

sessions["xgb_bot_proba"] = clf.predict_proba(X)[:, 1]

importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature importance:")
print(importances.to_string())

### The most interesting candidates: high probability but never flagged by UA

This is the list the User-Agent-only approach could never have produced.

In [ ]:
disguised_candidates = sessions[
    (sessions["known_bot_by_ua"] == 0) & (sessions["xgb_bot_proba"] > 0.7)
].sort_values("xgb_bot_proba", ascending=False)

print(f"Sessions with a \"clean\" UA but bot probability > 0.7: {len(disguised_candidates):,}")
disguised_candidates.head(20)[
    ["session_id", "n_hits", "duration_seconds", "interaction_ratio", "pages_per_hit",
     "xgb_bot_proba", "ecod_score", "useragent"]
]

## 8. Combining both signals and saving for review

**No automatic upload in this notebook.** The output is a review CSV: unlike User-Agent-based
detection (near-certainty), here we're working with statistical probabilities — the risk of a false
positive (a human with unusual behavior, e.g. an unstable mobile connection) is real and needs
validating before excluding that traffic from CJA metrics.

In [ ]:
sessions["flagged_behavioral_bot"] = (
    (sessions["ecod_is_anomaly"] == 1) | (sessions["xgb_bot_proba"] > 0.7)
) & (sessions["known_bot_by_ua"] == 0)  # only the NEW candidates; known ones are already handled upstream

review = sessions[sessions["flagged_behavioral_bot"]].sort_values("xgb_bot_proba", ascending=False)
print(f"Sessions for review: {len(review):,} out of {len(sessions):,} total")

review.to_csv("behavioral_bot_candidates.csv", index=False, encoding="utf-8")
print("Saved to behavioral_bot_candidates.csv")

review[["session_id", "visitor_id", "n_hits", "duration_seconds", "interaction_ratio",
        "xgb_bot_proba", "ecod_score", "useragent"]].head(30)

---

## Next steps

This notebook validates the logic on **one dataset and a few days**. Before extending it:

1. **Manual review** of `behavioral_bot_candidates.csv` — confirm these aren't false positives (e.g.
   users on an unstable connection, or a legitimate but unusual power-user session)
2. **Tune the thresholds** (ECOD's `contamination`, XGBoost's 0.7 probability cutoff) against
   observed results
3. **Add inter-hit time-delta variance** — probably the single strongest feature, left out here
   because it needs per-row timestamps, not just the session-level aggregate
4. **Evaluate ASN/datacenter enrichment**, if a usable signal is available from your
   CDN/WAF layer
5. Only then, **extend to all datasets and a wider window**, watching the compute cost of the window
   functions at larger volumes